# Gantt charts and schedules

Bars from a start to an end: a plan of tasks with dependencies and progress (`gantt`), and
jobs on machines that overlap in time (`Timeline.intervals`). Click a plot, then scroll to
zoom; hover a bar for its dates. Docs: <https://mluttikh.github.io/anytimeline/how-to/gantt.html>

In [ ]:
import numpy as np

from anytimeline import Timeline, gantt

## A project plan

A row per task: a lane per task in the table's order, dependencies as arrows, progress as a
darker fill, and milestones (start equal to end) as diamonds. `today=` draws a line.

In [ ]:
plan = {
    "task": ["Requirements", "Design", "Prototype", "Review", "Build backend",
             "Build frontend", "Write docs", "Test", "Release"],
    "start": np.array(["2026-09-01", "2026-09-08", "2026-09-15", "2026-09-29", "2026-09-30",
                       "2026-10-02", "2026-10-05", "2026-10-20", "2026-11-02"],
                      dtype="datetime64[D]"),
    "days": [7, 9, 14, 0, 18, 16, 20, 13, 0],
    "after": [None, "Requirements", "Design", "Prototype", "Review", "Review", "Design",
              "Build backend, Build frontend", "Test, Write docs"],
    "done": [1.0, 1.0, 1.0, 1.0, 0.6, 0.4, 0.3, 0.0, 0.0],
    "phase": ["plan", "plan", "plan", "plan", "build", "build", "build", "ship", "ship"],
}
gantt(
    plan, end=None, duration="days", duration_unit="d",
    depends="after", progress="done", color="phase", text="{task}",
    palette={"plan": "#4c78a8", "build": "#f58518", "ship": "#54a24b"},
    title="Project plan", today="2026-10-12",
)

## Sub-tasks

`parent=` puts tasks under the task they are part of. Click a task's name to fold it;
double-click to zoom to it.

In [ ]:
work = {
    "task": ["Backend", "API", "Database", "Frontend", "Pages", "Styles", "Release"],
    "part of": [None, "Backend", "Backend", None, "Frontend", "Frontend", None],
    "start": np.array(["2026-10-01", "2026-10-01", "2026-10-06", "2026-10-05", "2026-10-05",
                       "2026-10-12", "2026-10-23"], dtype="datetime64[D]"),
    "end": np.array(["2026-10-20", "2026-10-14", "2026-10-20", "2026-10-22", "2026-10-16",
                     "2026-10-22", "2026-10-23"], dtype="datetime64[D]"),
    "after": [None, None, None, None, None, None, "Backend, Frontend"],
}
gantt(work, parent="part of", depends="after", title="Work breakdown")

## Jobs on machines

`Timeline.intervals` draws bars on any lanes. Bars of a lane that overlap in time are
stacked in sub-rows, so none hides another. `marker` draws a line at a time.

In [ ]:
rng = np.random.default_rng(5)
n = 160
jobs = {
    "machine": rng.choice([f"press {i}" for i in range(1, 7)], n),
    "start": np.datetime64("2026-10-12T06:00", "m")
    + np.sort(rng.integers(0, 3 * 24 * 60, n)).astype("timedelta64[m]"),
    "minutes": rng.integers(20, 240, n),
    "product": rng.choice(["A", "B", "C", "D"], n),
}
schedule = Timeline(title="Jobs per machine", height=380)
schedule.intervals(jobs, end=None, duration="minutes", duration_unit="min", lane="machine",
                   color="product", text="{product}", tooltip=["minutes"])
schedule.marker(np.datetime64("2026-10-13T12:00"), "maintenance")
schedule

Select bars with the **Select** tool (drag a box), then read them back as rows of the table:

In [ ]:
schedule.selection